# SC2001 Project 2 - Part (a): Dijkstra with an Adjacency Matrix and an Array Priority Queue

**Task.** The input graph $G = (V, E)$ is stored in an **adjacency matrix** and the
priority queue is an **array**. Implement Dijkstra's algorithm in this setting and
analyse its time complexity with respect to $|V|$ and $|E|$, both theoretically and empirically.

**Contents**
1. Implementation
2. Theoretical analysis
3. Correctness check
4. Empirical analysis: vary $|V|$, then vary $|E|$
5. Conclusion

## Setup

In [ ]:
%matplotlib inline

import gc
import math
import random
import time

import matplotlib.pyplot as plt

INF = math.inf
SEED = 2001        # fixes the random graphs so the results are reproducible
MAX_WEIGHT = 100   # edge weights are random integers in [1, MAX_WEIGHT]

QUICK = False      # True = small sizes for a fast run (~20 s); False = full run (~1-2 min)

---
## 1. Implementation

**Data structures**

| Name | Meaning |
|---|---|
| `graph[u][v]` | weight of edge $(u, v)$, or `INF` if there is no edge ($|V| \times |V|$ matrix) |
| `d[v]` | current shortest-distance estimate of $v$; this array **is** the priority queue |
| `pi[v]` | predecessor of $v$ on the shortest path |
| `S[v]` | `True` once $v$'s distance is final, i.e. $v$ has been removed from the queue |

**Priority-queue operations on an array**
- *Extract-min*: scan all $|V|$ entries for the unvisited vertex with the smallest `d`, so $O(|V|)$.
- *Decrease-key*: overwrite `d[v]`, so $O(1)$.

**Finding the neighbours of $u$.** A matrix cannot list only $u$'s neighbours, so the whole
row `graph[u][0..|V|-1]` is scanned, which is $O(|V|)$ no matter how many edges $u$ has.

The function also counts its elementary operations. These counts are exact and free of
timing noise, so they test the analysis more directly than wall-clock time:
- `pq_scan`: array entries examined by extract-min
- `matrix_scan`: matrix cells examined while relaxing
- `decrease_key`: times a distance was improved

In [ ]:
def dijkstra_matrix_array(graph, source):
    # Single-source shortest paths: adjacency matrix + array priority queue.
    # Returns (d, pi, ops), where ops counts the elementary operations.
    n = len(graph)
    d = [INF] * n          # the priority queue: d[v] is v's priority
    pi = [None] * n
    S = [False] * n        # S[v] = True means v has left the queue
    d[source] = 0

    ops = {"pq_scan": 0, "matrix_scan": 0, "decrease_key": 0}

    for _ in range(n):
        # Extract-min: linear scan over the array, O(|V|)
        u = -1
        best = INF
        for v in range(n):
            ops["pq_scan"] += 1
            if not S[v] and d[v] < best:
                best = d[v]
                u = v
        if u == -1:        # every remaining vertex is unreachable
            break
        S[u] = True

        # Relax every edge out of u: scan the whole matrix row, O(|V|)
        row = graph[u]
        for v in range(n):
            ops["matrix_scan"] += 1
            w = row[v]
            if w != INF and not S[v] and d[u] + w < d[v]:
                d[v] = d[u] + w          # decrease-key on an array is O(1)
                pi[v] = u
                ops["decrease_key"] += 1

    return d, pi, ops

### Generating random graphs

`random_graph_edges(n, m)` returns exactly $m$ distinct directed edges on $n$ vertices,
with no self-loops. The first $n-1$ edges form a random tree rooted at vertex 0, so every
vertex is reachable from the source and Dijkstra always processes all $n$ vertices. The
remaining edges are sampled uniformly from the $n(n-1)$ possible pairs, so any $m$ from
$n-1$ (a tree) to $n(n-1)$ (a complete graph) can be generated.

In [ ]:
def random_graph_edges(n, m, rng):
    max_edges = n * (n - 1)
    if not n - 1 <= m <= max_edges:
        raise ValueError(f"need {n - 1} <= m <= {max_edges}, got {m}")

    # Random spanning tree rooted at 0: each vertex gets a parent placed before it
    order = list(range(1, n))
    rng.shuffle(order)
    order = [0] + order
    pairs = set()
    for i in range(1, n):
        pairs.add((order[rng.randrange(i)], order[i]))

    # Sample the remaining edges by index into the n(n-1) off-diagonal pairs.
    # At most n-1 samples can collide with tree edges, so oversample by n-1.
    extra = m - (n - 1)
    k = min(max_edges, extra + n - 1)
    for idx in rng.sample(range(max_edges), k):
        if len(pairs) == m:
            break
        u, r = divmod(idx, n - 1)
        v = r if r < u else r + 1     # skip the diagonal (no self-loops)
        pairs.add((u, v))

    return [(u, v, rng.randint(1, MAX_WEIGHT)) for u, v in pairs]


def to_matrix(n, edges):
    graph = [[INF] * n for _ in range(n)]
    for u, v, w in edges:
        graph[u][v] = w
    return graph

---
## 2. Theoretical analysis

| Step | Work per execution | Times executed | Total |
|---|---|---|---|
| Initialise `d`, `pi`, `S` | $O(1)$ per vertex | $|V|$ | $O(|V|)$ |
| Extract-min (scan the array) | $O(|V|)$ | $|V|$ | $O(|V|^2)$ |
| Scan row `graph[u]` for neighbours | $O(|V|)$ | $|V|$ | $O(|V|^2)$ |
| Decrease-key (overwrite `d[v]`) | $O(1)$ | at most $|E|$ | $O(|E|)$ |

$$
T(|V|, |E|) = O(|V|) + O(|V|^2) + O(|V|^2) + O(|E|) = O(|V|^2 + |E|) = O(|V|^2)
$$

because a simple directed graph has $|E| \le |V|(|V|-1) < |V|^2$.

**The bound is tight in every case.** Both $|V|^2$ terms are paid in full whatever the
input: each extract-min scans all $|V|$ array entries, and each row scan reads all $|V|$
cells, including the empty ones. So the running time is $\Theta(|V|^2)$ in the best,
average and worst case. Exactly $|V|^2$ array entries and $|V|^2$ matrix cells are
examined, $2|V|^2$ in total.

**Dependence on $|E|$.** $|E|$ only affects how many decrease-keys happen, and each costs
$O(1)$. The cost is therefore asymptotically independent of $|E|$: a tree with $|V|-1$ edges
costs the same $\Theta(|V|^2)$ as a complete graph with $|V|(|V|-1)$ edges.

**Predictions to test empirically**
1. Running time grows as $|V|^2$, so time $/\,|V|^2$ stays roughly constant.
2. The operation count `pq_scan + matrix_scan` is exactly $2|V|^2$.
3. With $|V|$ fixed, changing $|E|$ leaves the operation count unchanged and the time nearly flat.

---
## 3. Correctness check

1. The textbook example from CLRS (Fig. 24.6), with vertices $s, t, x, y, z = 0, 1, 2, 3, 4$
2. A graph with an unreachable vertex, whose distance must stay $\infty$
3. 200 random graphs, checked against Bellman-Ford, an independent $O(|V||E|)$ algorithm

In [ ]:
def bellman_ford(n, edges, source):
    # Independent reference algorithm, used only to check the answers
    d = [INF] * n
    d[source] = 0
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if d[u] + w < d[v]:
                d[v] = d[u] + w
                changed = True
        if not changed:
            break
    return d


# 1. Textbook example
edges = [(0, 1, 10), (0, 3, 5), (1, 2, 1), (1, 3, 2), (2, 4, 4),
         (3, 1, 3), (3, 2, 9), (3, 4, 2), (4, 0, 7), (4, 2, 6)]
d, pi, _ = dijkstra_matrix_array(to_matrix(5, edges), 0)
print("textbook example  d  =", d)
print("                  pi =", pi)
assert d == [0, 8, 9, 5, 7]
assert pi == [None, 3, 1, 0, 3]

# 2. Unreachable vertex
d, _, _ = dijkstra_matrix_array(to_matrix(3, [(0, 1, 4)]), 0)
print("unreachable case  d  =", d)
assert d == [0, 4, INF]

# 3. Random graphs against Bellman-Ford
rng = random.Random(SEED)
for _ in range(200):
    n = rng.randint(2, 40)
    m = rng.randint(n - 1, n * (n - 1))
    edges = random_graph_edges(n, m, rng)
    d, _, _ = dijkstra_matrix_array(to_matrix(n, edges), 0)
    assert d == bellman_ford(n, edges, 0)

print("\nAll correctness checks pass (textbook example, unreachable case, 200 random graphs).")

---
## 4. Empirical analysis

`measure` builds one random graph, then runs Dijkstra `reps` times and keeps the fastest
time. Building the graph is not timed. The garbage collector is paused while timing so its
pauses don't add noise.

`log_slope` fits $\log(\text{time}) = k \log|V| + c$ by least squares. The slope $k$ is the
measured exponent in time $\propto |V|^k$, and the theory predicts $k \approx 2$.

In [ ]:
def measure(n, m, reps, rng):
    graph = to_matrix(n, random_graph_edges(n, m, rng))
    best = INF
    gc.disable()
    try:
        for _ in range(reps):
            start = time.perf_counter()
            _, _, ops = dijkstra_matrix_array(graph, 0)
            best = min(best, time.perf_counter() - start)
    finally:
        gc.enable()
    return best, ops


def log_slope(xs, ys):
    lx = [math.log(x) for x in xs]
    ly = [math.log(y) for y in ys]
    mx, my = sum(lx) / len(lx), sum(ly) / len(ly)
    num = sum((a - mx) * (b - my) for a, b in zip(lx, ly))
    den = sum((a - mx) ** 2 for a in lx)
    return num / den


if QUICK:
    SIZES, REPS, FIXED_V, E_STEPS = list(range(100, 801, 100)), 2, 500, 6
else:
    SIZES, REPS, FIXED_V, E_STEPS = list(range(200, 2001, 200)), 3, 1500, 8

### 4.1 Varying $|V|$

$|V|$ runs over a range of sizes at two densities:
- **sparse**: $|E| = 5|V|$
- **dense**: $|E| = |V|(|V|-1)/2$, half of all possible edges

If the analysis is right, both densities grow as $|V|^2$ and take about the same time.

In [ ]:
rng = random.Random(SEED)
vary_v = {"sparse": [], "dense": []}

header = (f"{'|V|':>6} {'density':>7} {'|E|':>9} {'time (s)':>9} {'t/|V|^2 (ns)':>13} "
          f"{'pq_scan':>10} {'matrix_scan':>11} {'dec_key':>8}")
print(header)
print("-" * len(header))
for n in SIZES:
    for label, m in (("sparse", 5 * n), ("dense", n * (n - 1) // 2)):
        t, ops = measure(n, m, REPS, rng)
        vary_v[label].append((n, m, t, ops))
        print(f"{n:>6} {label:>7} {m:>9} {t:>9.4f} {t / n**2 * 1e9:>13.2f} "
              f"{ops['pq_scan']:>10} {ops['matrix_scan']:>11} {ops['decrease_key']:>8}")

print()
for label, rows in vary_v.items():
    k = log_slope([r[0] for r in rows], [r[2] for r in rows])
    exact = all(r[3]["pq_scan"] + r[3]["matrix_scan"] == 2 * r[0] ** 2 for r in rows)
    print(f"{label:>6}: fitted exponent k = {k:.2f};  scans == 2|V|^2 for every size: {exact}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ax = axes[0]
for label, marker in (("sparse", "o"), ("dense", "s")):
    rows = vary_v[label]
    ax.plot([r[0] for r in rows], [r[2] for r in rows], marker=marker, label=f"{label} graph")
ax.set(title="Running time vs |V|", xlabel="|V|", ylabel="time (s)")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for label, marker in (("sparse", "o"), ("dense", "s")):
    rows = vary_v[label]
    ax.plot([r[0] for r in rows], [r[2] / r[0] ** 2 * 1e9 for r in rows],
            marker=marker, label=f"{label} graph")
ax.set(title="time / |V|$^2$  (flat line = $\\Theta(|V|^2)$)", xlabel="|V|", ylabel="ns per |V|$^2$")
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(alpha=0.3)

ax = axes[2]
ns = [r[0] for r in vary_v["dense"]]
for label, style in (("dense", "s-"), ("sparse", "o--")):
    ax.plot(ns, [r[3]["pq_scan"] + r[3]["matrix_scan"] for r in vary_v[label]], style,
            label=f"cells scanned ({label})")
ax.plot(ns, [2 * n * n for n in ns], ":", color="black", label="2|V|$^2$")
ax.set(title="Operation count vs |V|", xlabel="|V|", ylabel="operations")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

**Observations**
- **Time grows quadratically.** The fitted exponents are close to 2. Time $/\,|V|^2$ stays
  roughly flat across sizes, which is the signature of $\Theta(|V|^2)$.
- **Operation counts match exactly.** `pq_scan` and `matrix_scan` are each exactly $|V|^2$,
  so their sum is $2|V|^2$ for both densities. The sparse and dense curves lie on top of
  each other and on the $2|V|^2$ line.
- **Decrease-key is negligible.** Even on dense graphs it happens only a few thousand
  times, compared with millions of scans.
- **Dense graphs are slightly slower, by a constant factor only.** When a cell is empty,
  the condition `w != INF` fails immediately and the loop moves on. When there is an edge,
  it also checks `S[v]` and computes `d[u] + w`. Each cell therefore costs a little more on a
  dense graph. This raises the constant in front of $|V|^2$ (the two flat lines in the
  middle plot sit at different heights) but not the growth rate.

### 4.2 Varying $|E|$ with $|V|$ fixed

$|V|$ is held fixed while $|E|$ goes from $|V|-1$ (a tree) to $|V|(|V|-1)$ (a complete
directed graph). The theory predicts the same operation count throughout and a nearly
flat running time.

In [ ]:
rng = random.Random(SEED + 1)
max_m = FIXED_V * (FIXED_V - 1)
edge_counts = sorted({FIXED_V - 1} | {round(max_m * i / E_STEPS) for i in range(1, E_STEPS + 1)})
vary_e = []

print(f"|V| = {FIXED_V} fixed\n")
header = f"{'|E|':>9} {'density':>8} {'time (s)':>9} {'pq_scan':>10} {'matrix_scan':>11} {'dec_key':>8}"
print(header)
print("-" * len(header))
for m in edge_counts:
    t, ops = measure(FIXED_V, m, REPS, rng)
    vary_e.append((m, t, ops))
    print(f"{m:>9} {m / max_m:>8.1%} {t:>9.4f} {ops['pq_scan']:>10} "
          f"{ops['matrix_scan']:>11} {ops['decrease_key']:>8}")

times = [r[1] for r in vary_e]
print(f"\n|E| grew by a factor of {edge_counts[-1] / edge_counts[0]:.0f}, "
      f"time grew by a factor of {max(times) / min(times):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
es = [r[0] for r in vary_e]

ax = axes[0]
ax.plot(es, [r[1] for r in vary_e], "o-")
ax.set(title=f"Running time vs |E|  (|V| = {FIXED_V})", xlabel="|E|", ylabel="time (s)")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(es, [r[2]["pq_scan"] + r[2]["matrix_scan"] for r in vary_e], "o-", label="cells scanned")
ax.plot(es, [r[2]["decrease_key"] for r in vary_e], "s-", label="decrease-key")
ax.axhline(2 * FIXED_V ** 2, ls=":", color="black", label="2|V|$^2$")
ax.set(title=f"Operation count vs |E|  (|V| = {FIXED_V})", xlabel="|E|", ylabel="operations (log scale)")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

**Observations**
- **The scan count does not depend on $|E|$.** It is exactly $2|V|^2$ for every value of
  $|E|$, from a tree up to a complete graph.
- **Time is almost flat.** $|E|$ grows by several orders of magnitude (see the printed factor), but the time rises by
  well under 2×. The small, roughly linear rise is the per-cell constant described in 4.1:
  cells that hold an edge cost a little more than empty ones. It is bounded by a constant
  factor, because every cell is visited exactly once either way.
- **Decrease-keys level off.** Their count rises quickly, then plateaus at a few thousand,
  far below $|E|$. A vertex's distance is improved only a few times on average, and the
  number of improvements is capped at $|E|$ by the theory.

---
## 5. Conclusion

With an adjacency matrix and an array priority queue, Dijkstra's algorithm runs in

$$\Theta(|V|^2)$$

in the best, average and worst case, independently of $|E|$.

- **Theory.** There are $|V|$ extract-mins, each an $O(|V|)$ array scan, and $|V|$ row
  scans, each $O(|V|)$. That gives $2|V|^2$ cell reads. The $\le |E|$ decrease-keys add
  only $O(|E|) \subseteq O(|V|^2)$.
- **Experiments.**
  - The operation count is exactly $2|V|^2$ for every graph tested.
  - The measured running-time exponent is about 2.
  - Holding $|V|$ fixed while increasing $|E|$ by three orders of magnitude changes the
    time only by a small constant factor.

**Implication.** This implementation pays the full $|V|^2$ even on a very sparse graph, but
nothing extra on a complete one. It therefore suits dense graphs, where $|E| = \Theta(|V|^2)$
and no algorithm can avoid reading the edges anyway. On sparse graphs most of its work is
spent scanning empty matrix cells.